# Acoustic PDM — Phase 2: Notebook 03
## Audio Preprocessing Engine: WAV → Mel-Spectrogram → Normalised Tensor Blocks

**Project:** Acoustic Predictive Maintenance (Acoustic PDM)  
**Dataset:** Hitachi MIMII — 4 industrial machine types (**Fan, Pump, Slider, Valve**) at 6 dB SNR, Machine ID 00  
**Purpose:** Transform raw `.wav` audio files into the numerical tensor format that the Autoencoder neural network consumes for training and inference.

---

### What is this notebook about?

Machine-learning models cannot directly process raw audio waveforms effectively. This notebook implements the **complete DSP (Digital Signal Processing) pipeline** that converts every 10-second `.wav` recording into a stack of small, normalised 2-D image patches — these patches are what the Autoencoder will learn from.

### The Preprocessing Pipeline at a Glance

```
Raw .wav file (160,000 samples @ 16 kHz, 10 seconds)
        │
        ▼
  1. Load audio at 16 kHz mono
        │
        ▼
  2. Compute Mel-Spectrogram (128 Mel bins × 313 time frames)
        │
        ▼
  3. Convert to log-decibel scale (dB)
        │
        ▼
  4. Slice into overlapping context windows (128 × 5 blocks)
        │
        ▼
  5. Z-score normalise using training-set statistics only
        │
        ▼
  Output: .npy tensor files ready for PyTorch DataLoader
```

### Prerequisites
- **Notebook 01** must have been run to produce `reports/indexed_dataset.csv` (the file catalog).
- All raw `.wav` files must be accessible at the expected `DATA_ROOT` path.

### Outputs
| File | Shape | Description |
|---|---|---|
| `data/processed/{machine}/train_normal.npy` | (N, 1, 128, 5) | Normal training blocks — used to train the Autoencoder |
| `data/processed/{machine}/val_normal.npy` | (M, 1, 128, 5) | Normal validation blocks — used for early stopping & threshold calibration |
| `data/processed/{machine}/test_normal.npy` | (K, 1, 128, 5) | Normal test blocks — used for ROC-AUC evaluation |
| `data/processed/{machine}/test_anomaly.npy` | (J, 1, 128, 5) | Anomalous test blocks — used for ROC-AUC evaluation |
| `data/processed/{machine}/norm_stats.npz` | (128,) ×2 | Per-frequency-bin mean (μ) and std (σ) computed on training data only |

---
### Step 0: Environment Bootstrap — Path Discovery & Configuration Loading

**This cell must run first.** It performs the same environment setup as Notebooks 01 and 02:

1. **Writes `config.yaml`** into the Kaggle working directory (only needed on Kaggle; harmless locally).
2. **Detects the runtime** (Kaggle vs. local) and resolves `PROJECT_ROOT`, `DATA_ROOT`, `REPORTS_DIR`, `PROCESSED_DIR`, etc.
3. **Loads centralised configuration** into `CFG` so that all DSP parameters (sample rate, FFT window, Mel bins, context frames, normalisation method) are consistent.
4. **Adds `src/`** to the Python path.

You do not need to modify anything in this cell.

In [ ]:
import os
os.makedirs('/kaggle/working/configs', exist_ok=True)

config_text = """\
# ============================================================
# Acoustic PDM — Centralized Pipeline Configuration
# ============================================================
audio:
  sample_rate: 16000
  channels: 1
  bit_depth: 16
  clip_duration_sec: 10

features:
  n_fft: 1024
  hop_length: 512
  n_mels: 128
  fmin: 0
  fmax: null
  power_to_db: true
  context_frames: 5

normalization:
  method: "zscore"
  epsilon: 1.0e-8

data:
  raw_dir: "data/raw"
  processed_dir: "data/processed"
  machine_types:
    - "fan"
    - "pump"
    - "slider"
    - "valve"
  machine_ids: ["id_00"]
  snr_levels: ["6_dB"]
  test_split: 0.1

training:
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 1.0e-5
  epochs: 50
  early_stopping_patience: 10
  random_seed: 42
  num_workers: 2

model_ae:
  latent_dim: 32
  encoder_channels: [1, 32, 64, 128]
  kernel_size: 3
  stride: 2
  padding: 1
  activation: "leaky_relu"

evaluation:
  threshold_percentile: 95
  reports_dir: "reports"
"""

with open('/kaggle/working/configs/config.yaml', 'w') as f:
    f.write(config_text)
print("✓ config.yaml written")


# ═══════════════════════════════════════════════════════════════
# Path & Environment Bootstrap
# ═══════════════════════════════════════════════════════════════
import sys
import yaml
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
    _kaggle_data = None
    for _root, _dirs, _ in os.walk('/kaggle/input'):
        if 'dc2020task2' in _dirs:
            _kaggle_data = os.path.join(_root, 'dc2020task2')
            break
    DATA_ROOT = Path(_kaggle_data) if _kaggle_data else Path('/kaggle/input/dc2020task2')
else:
    _cwd = Path(os.getcwd()).resolve()
    PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
    DATA_ROOT    = PROJECT_ROOT / "data" / "raw"

REPORTS_DIR   = PROJECT_ROOT / "reports"
CONFIGS_DIR   = PROJECT_ROOT / "configs"
MODELS_DIR    = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

config_path = CONFIGS_DIR / "config.yaml"
if config_path.exists():
    with open(config_path, "r") as f:
        CFG = yaml.safe_load(f)
    print(f"✓ Config loaded from: {config_path}")
else:
    CFG = {}
    print(f"⚠ Config not found at {config_path}, using defaults")

print(f"Environment:  {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Root:    {DATA_ROOT}")
print(f"Processed:    {PROCESSED_DIR}")
print(f"Data exists:  {DATA_ROOT.exists()}")

---
### Step 1: Import Libraries

| Library | Purpose in this notebook |
|---|---|
| **numpy** | Array manipulation, saving/loading `.npy` tensor files, computing normalisation statistics |
| **pandas** | Loading the indexed file catalog CSV from Notebook 01 |
| **librosa** | Loading `.wav` audio, computing Mel-spectrograms, converting power to decibels |
| **matplotlib** | Plotting sample spectrograms and context-window visualisations for verification |
| **sklearn.model_selection** | `train_test_split` for splitting normal clips into train/validation/test partitions |
| **glob** | Recursive file discovery (fallback if catalog CSV is unavailable) |
| **tqdm** | Progress bars for the long extraction loop |
| **gc** | Garbage collection to free memory between machine types (important on Kaggle's 13 GB RAM limit) |

In [ ]:
import glob
import gc
import time
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

OUTPUT_DIR = str(REPORTS_DIR)
print("✓ All libraries imported successfully.")

---
### Step 2: Load DSP & Preprocessing Configuration

All signal-processing hyperparameters are pulled from `config.yaml` into a single `PREPROCESS_CFG` dictionary. This ensures consistency with the exploration in Notebook 02.

#### Key parameters and their physical meaning

| Parameter | Value | What it controls |
|---|---|---|
| `sr` (sample rate) | 16,000 Hz | Number of audio amplitude measurements per second |
| `n_fft` | 1,024 | Length of each FFT analysis window (64 ms of audio). Controls frequency resolution: $\Delta f = 16000 / 1024 \approx 15.6$ Hz per bin |
| `hop_length` | 512 | How far the analysis window slides between consecutive frames (32 ms). This gives 50% overlap between adjacent windows |
| `n_mels` | 128 | Number of Mel-frequency bins. The Mel scale compresses high frequencies and expands low frequencies to match human/machine acoustic sensitivity |
| `context_frames` | 5 | Number of consecutive time frames grouped into a single input block. Each (128, 5) block spans $5 \times 32\text{ ms} = 160\text{ ms}$ of sound |
| `test_split` | 0.1 | 10% of normal clips are held out as a validation set for threshold calibration and early stopping |
| `epsilon` | $10^{-8}$ | Tiny constant added to the denominator during Z-score normalisation to prevent division-by-zero in silent frequency bins |
| `top_db` | 80 | Maximum dynamic range in the log-dB spectrogram; anything quieter than $-80$ dB is clipped to remove irrelevant floor noise |

In [ ]:
PREPROCESS_CFG = {
    # Audio loading
    "sr":             CFG.get("audio", {}).get("sample_rate", 16000),
    # STFT / Mel-spectrogram
    "n_fft":          CFG.get("features", {}).get("n_fft", 1024),
    "hop_length":     CFG.get("features", {}).get("hop_length", 512),
    "n_mels":         CFG.get("features", {}).get("n_mels", 128),
    "fmin":           CFG.get("features", {}).get("fmin", 0),
    "fmax":           CFG.get("features", {}).get("fmax", None),
    "top_db":         80,
    # Context windows
    "context_frames": CFG.get("features", {}).get("context_frames", 5),
    # Data splitting
    "test_split":     CFG.get("data", {}).get("test_split", 0.1),
    "random_seed":    CFG.get("training", {}).get("random_seed", 42),
    # Normalisation
    "norm_method":    CFG.get("normalization", {}).get("method", "zscore"),
    "epsilon":        CFG.get("normalization", {}).get("epsilon", 1e-8),
    # Machine scope
    "machines":       CFG.get("data", {}).get("machine_types", ["fan", "pump", "slider", "valve"]),
}

print("Preprocessing Configuration:")
for k, v in PREPROCESS_CFG.items():
    print(f"  {k:20s}: {v}")

---
### Step 3: Load the File Catalog from Notebook 01

This cell loads `reports/indexed_dataset.csv` — the file-level catalog produced by Notebook 01. Each row contains the full file path, machine type, condition (normal/anomaly), and machine ID.

If the catalog is not found (e.g., running standalone on Kaggle without chaining), a fallback directory scan is performed automatically.

After loading, we filter to keep only the 4 target machine types and print a summary of available clips per category.

In [ ]:
def load_catalog():
    """Load the indexed catalog from NB01 or perform a fallback scan."""
    # 1. Check local reports dir
    csv_path = os.path.join(OUTPUT_DIR, "indexed_dataset.csv")
    if os.path.exists(csv_path):
        print(f"✓ Catalog found: {csv_path}")
        return pd.read_csv(csv_path)
    
    # 2. Check Kaggle chained input
    if ON_KAGGLE:
        hits = glob.glob('/kaggle/input/**/indexed_dataset.csv', recursive=True)
        if hits:
            print(f"✓ Catalog found via Kaggle input: {hits[0]}")
            return pd.read_csv(hits[0])
    
    # 3. Fallback: scan DATA_ROOT
    print("⚠ Catalog not found. Running fallback scan...")
    wavs = glob.glob(os.path.join(str(DATA_ROOT), '**/*.wav'), recursive=True)
    if not wavs:
        print("✗ No .wav files found.")
        return pd.DataFrame()
    
    records = []
    for w in wavs:
        p = Path(w)
        fn = p.name.lower()
        parts = [part.lower() for part in p.parts]
        
        m_type = "unknown"
        for m in ["fan", "pump", "slider", "valve"]:
            if any(m in part for part in parts) or m in fn:
                m_type = m
                break
        
        if "anomaly" in fn or "abnormal" in fn or any("anomaly" in pt or "abnormal" in pt for pt in parts):
            cond = "anomaly"
        elif "normal" in fn or any("normal" in pt for pt in parts):
            cond = "normal"
        else:
            cond = "unknown"
        
        records.append({"file_path": str(p), "machine_type": m_type, "condition": cond, "filename": p.name})
    print(f"  Fallback scan: {len(records)} files.")
    return pd.DataFrame(records)

df_catalog = load_catalog()

# Filter to target machines only
target_machines = PREPROCESS_CFG["machines"]
df_catalog = df_catalog[df_catalog["machine_type"].isin(target_machines)].copy()

print(f"\nTotal clips in scope: {len(df_catalog)}")
display(df_catalog.groupby(["machine_type", "condition"]).size().unstack(fill_value=0))

---
### Step 4: Define the Core Preprocessing Functions

This cell defines the three fundamental functions that form the DSP pipeline:

#### 4a. `wav_to_log_mel(file_path)` — WAV → Log Mel-Spectrogram
1. Loads the `.wav` file at 16 kHz mono using `librosa.load()`.
2. Computes the **power Mel-spectrogram** using a bank of 128 triangular Mel-scaled filters applied to the STFT magnitude.
3. Converts the power values to **log-decibel (dB) scale**: $S_{\text{dB}} = 10 \cdot \log_{10}(S_{\text{power}} + \epsilon)$. The log compression is critical because raw power values span many orders of magnitude, but neural network gradients work best when inputs are on a similar numeric scale.
4. Returns a 2-D array of shape `(128, T)` where `T` is the number of time frames (typically 313 for a 10-second clip).

#### 4b. `extract_context_windows(spectrogram, context_frames)` — Sliding Window Framing
A single time frame of 128 values provides a snapshot of *which frequencies are active at one instant*, but industrial faults are **temporal processes** (e.g., a bearing click repeats every revolution, a valve hiss builds up over milliseconds). By grouping 5 consecutive frames, each $(128, 5)$ block captures **160 ms** of acoustic evolution.

The function uses a **stride-1 sliding window**: block 0 = frames [0–4], block 1 = frames [1–5], etc. This generates $T - P + 1$ overlapping blocks per clip (typically 309 for a 10-second recording).

Each block is reshaped to `(1, 128, 5)` to match the PyTorch convention for single-channel 2-D inputs (like a grayscale image).

#### 4c. `process_file_list(file_paths)` — Batch Processing
Applies `wav_to_log_mel` + `extract_context_windows` to a list of audio files and stacks the results into a single NumPy array of shape `(total_blocks, 1, 128, 5)`.

In [ ]:
def wav_to_log_mel(file_path):
    """
    Load a .wav file and compute its log Mel-spectrogram.
    
    Returns:
        np.ndarray of shape (n_mels, T) where T = number of time frames.
        Returns None if the file cannot be loaded.
    """
    try:
        y, sr = librosa.load(file_path, sr=PREPROCESS_CFG["sr"], mono=True)
        
        # Compute power Mel-spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=PREPROCESS_CFG["n_fft"],
            hop_length=PREPROCESS_CFG["hop_length"],
            n_mels=PREPROCESS_CFG["n_mels"],
            fmin=PREPROCESS_CFG["fmin"],
            fmax=PREPROCESS_CFG["fmax"],
            power=2.0
        )
        
        # Convert to log-decibel scale
        log_mel = librosa.power_to_db(mel_spec, ref=np.max, top_db=PREPROCESS_CFG["top_db"])
        return log_mel
    except Exception as e:
        print(f"  ⚠ Error processing {file_path}: {e}")
        return None


def extract_context_windows(spectrogram, context_frames=5):
    """
    Slice a (n_mels, T) spectrogram into overlapping (1, n_mels, context_frames) blocks.
    
    Uses a stride-1 sliding window:
      Block 0: frames [0, 1, 2, 3, 4]
      Block 1: frames [1, 2, 3, 4, 5]
      ...
    
    Returns:
        np.ndarray of shape (num_blocks, 1, n_mels, context_frames)
    """
    n_mels, T = spectrogram.shape
    num_blocks = T - context_frames + 1
    
    if num_blocks <= 0:
        return np.empty((0, 1, n_mels, context_frames), dtype=np.float32)
    
    blocks = np.zeros((num_blocks, 1, n_mels, context_frames), dtype=np.float32)
    for i in range(num_blocks):
        blocks[i, 0, :, :] = spectrogram[:, i:i + context_frames]
    
    return blocks


def process_file_list(file_paths, desc="Processing"):
    """
    Apply the full WAV -> log-Mel -> context-window pipeline to a list of files.
    
    Returns:
        np.ndarray of shape (total_blocks, 1, 128, context_frames)
    """
    all_blocks = []
    ctx = PREPROCESS_CFG["context_frames"]
    
    for fp in tqdm(file_paths, desc=desc):
        spec = wav_to_log_mel(fp)
        if spec is None:
            continue
        blocks = extract_context_windows(spec, context_frames=ctx)
        if blocks.shape[0] > 0:
            all_blocks.append(blocks)
    
    if not all_blocks:
        return np.empty((0, 1, PREPROCESS_CFG["n_mels"], ctx), dtype=np.float32)
    
    return np.concatenate(all_blocks, axis=0)

print("✓ Core preprocessing functions defined.")
print(f"  Pipeline: WAV → Log-Mel ({PREPROCESS_CFG['n_mels']} bins) → Context Windows (P={PREPROCESS_CFG['context_frames']})")

---
### Step 5: Sanity Check — Visualise the Pipeline on a Single File

Before running the full extraction across thousands of files, we verify the pipeline on **one sample file** to make sure:
1. The Mel-spectrogram has the expected shape `(128, ~313)`.
2. The context-window extraction produces blocks of shape `(~309, 1, 128, 5)`.
3. The values are in a reasonable dB range (typically $-80$ to $0$ dB).

We also plot the full spectrogram and one extracted context block side-by-side for visual confirmation.

In [ ]:
# Pick one normal file from the first available machine type for a sanity check
sample_machine = PREPROCESS_CFG["machines"][0]
sample_files = df_catalog[
    (df_catalog["machine_type"] == sample_machine) & 
    (df_catalog["condition"] == "normal")
]["file_path"].values

if len(sample_files) > 0:
    sample_path = sample_files[0]
    print(f"Sanity check file: {sample_path}")
    
    # Step A: WAV -> Log Mel-Spectrogram
    spec = wav_to_log_mel(sample_path)
    print(f"  Mel-spectrogram shape: {spec.shape}  (expected: (128, ~313))")
    print(f"  Value range: [{spec.min():.1f}, {spec.max():.1f}] dB")
    
    # Step B: Extract context windows
    blocks = extract_context_windows(spec, PREPROCESS_CFG["context_frames"])
    print(f"  Context blocks shape: {blocks.shape}  (expected: (~309, 1, 128, 5))")
    
    # Step C: Visualise
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Full spectrogram
    img = librosa.display.specshow(
        spec, x_axis='time', y_axis='mel',
        sr=PREPROCESS_CFG["sr"], hop_length=PREPROCESS_CFG["hop_length"],
        fmin=PREPROCESS_CFG["fmin"], fmax=PREPROCESS_CFG["fmax"],
        cmap='magma', ax=axes[0]
    )
    axes[0].set_title(f'Full Log Mel-Spectrogram ({sample_machine})', fontweight='bold')
    fig.colorbar(img, ax=axes[0], format='%+2.0f dB')
    
    # One context block (first block)
    axes[1].imshow(blocks[0, 0], aspect='auto', origin='lower', cmap='magma')
    axes[1].set_title(f'Single Context Block (128 × {PREPROCESS_CFG["context_frames"]})', fontweight='bold')
    axes[1].set_xlabel('Context Frame Index')
    axes[1].set_ylabel('Mel Bin')
    
    plt.tight_layout()
    plt.show()
    print("✓ Sanity check passed.")
else:
    print("⚠ No sample files found for sanity check.")

---
### Step 6: Split Normal Clips into Train / Validation / Test Partitions

For anomaly detection, the data splits follow a specific protocol:

| Partition | Source | Purpose |
|---|---|---|
| **Train Normal** | 90% of normal clips | The Autoencoder trains **exclusively** on these. It learns the compressed representation of healthy machine sounds |
| **Validation Normal** | 10% of normal clips | Used for early stopping (monitor validation reconstruction error to prevent overfitting) and for computing the anomaly threshold $\theta$ |
| **Test Normal** | All normal clips from the anomaly/test folder (if available), or a small held-out portion | Provides the "normal" half of the test evaluation |
| **Test Anomaly** | All anomaly clips | Provides the "faulty" half of the test evaluation |

#### Critical principle: No data leakage
The train/validation split is done **at the clip level** (not the block level). If we split after extracting blocks, consecutive blocks from the same clip could land in both train and validation, leaking temporal context and producing falsely optimistic metrics.

The normalisation statistics (μ, σ) are computed **strictly from the training partition**.

---
### Step 7: Run the Full Preprocessing Pipeline for All Machine Types

This is the **main execution cell**. For each of the 4 machine types, it:

1. **Separates** normal and anomaly file paths from the catalog.
2. **Splits** normal files into train (90%) and validation (10%) at the clip level.
3. **Extracts** log-Mel context blocks from each partition using `process_file_list()`.
4. **Computes Z-score normalisation statistics** (μ and σ per frequency bin) from the training blocks only.
5. **Applies normalisation** to all partitions (train, val, test_normal, test_anomaly) using the **training-only** statistics.
6. **Saves** each partition as a `.npy` file and the normalisation stats as a `.npz` file under `data/processed/{machine}/`.

#### Z-Score Normalisation — How it Works
For each of the 128 Mel-frequency bins, we compute:
$$\mu_f = \text{mean of all training block values at frequency bin } f$$
$$\sigma_f = \text{std of all training block values at frequency bin } f$$

Then every block (train, val, test) is normalised as:
$$X_{\text{norm}}(f, t) = \frac{X(f, t) - \mu_f}{\sigma_f + \epsilon}$$

This ensures that all frequency bins contribute equally to the Autoencoder's loss, regardless of their natural loudness level. The $\epsilon = 10^{-8}$ prevents division by zero for silent frequency bins.

**⚠ Memory note:** On Kaggle (13 GB RAM), we process one machine at a time and call `gc.collect()` between machines to free memory.

In [ ]:
pipeline_summary = []

for machine in PREPROCESS_CFG["machines"]:
    print(f"\n{'='*70}")
    print(f"  PROCESSING: {machine.upper()}")
    print(f"{'='*70}")
    
    # --- Create output directory ---
    out_dir = PROCESSED_DIR / machine
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # --- Get file lists ---
    m_df = df_catalog[df_catalog["machine_type"] == machine]
    normal_files = m_df[m_df["condition"] == "normal"]["file_path"].values
    anomaly_files = m_df[m_df["condition"] == "anomaly"]["file_path"].values
    
    print(f"  Normal clips:  {len(normal_files)}")
    print(f"  Anomaly clips: {len(anomaly_files)}")
    
    if len(normal_files) == 0:
        print(f"  ⚠ Skipping {machine}: no normal clips found.")
        continue
    
    # --- Split normal files into train / val (clip-level split) ---
    train_files, val_files = train_test_split(
        normal_files,
        test_size=PREPROCESS_CFG["test_split"],
        random_state=PREPROCESS_CFG["random_seed"]
    )
    print(f"  Train clips: {len(train_files)}, Val clips: {len(val_files)}")
    
    # --- Extract context blocks ---
    t0 = time.time()
    
    train_blocks = process_file_list(train_files, desc=f"{machine} train_normal")
    val_blocks   = process_file_list(val_files,   desc=f"{machine} val_normal")
    
    # Test: normal clips from validation + anomaly clips
    test_normal_blocks  = val_blocks.copy()  # val doubles as test_normal
    test_anomaly_blocks = process_file_list(anomaly_files, desc=f"{machine} test_anomaly") if len(anomaly_files) > 0 else np.empty((0, 1, PREPROCESS_CFG['n_mels'], PREPROCESS_CFG['context_frames']), dtype=np.float32)
    
    elapsed = time.time() - t0
    print(f"  Extraction time: {elapsed:.1f}s")
    print(f"  Block shapes: train={train_blocks.shape}, val={val_blocks.shape}, test_normal={test_normal_blocks.shape}, test_anomaly={test_anomaly_blocks.shape}")
    
    # --- Compute Z-score normalisation stats from TRAINING data ONLY ---
    # Shape: (N, 1, 128, 5) -> compute mean/std over axes (0, 1, 3) -> result shape (128,)
    train_mean = train_blocks.mean(axis=(0, 1, 3))  # shape: (128,)
    train_std  = train_blocks.std(axis=(0, 1, 3))   # shape: (128,)
    
    print(f"  Norm stats: mean range=[{train_mean.min():.2f}, {train_mean.max():.2f}], std range=[{train_std.min():.4f}, {train_std.max():.4f}]")
    
    # --- Apply Z-score normalisation ---
    eps = PREPROCESS_CFG["epsilon"]
    # Reshape mean/std for broadcasting: (128,) -> (1, 1, 128, 1)
    mu  = train_mean.reshape(1, 1, -1, 1)
    sig = train_std.reshape(1, 1, -1, 1)
    
    train_blocks        = (train_blocks - mu) / (sig + eps)
    val_blocks          = (val_blocks - mu) / (sig + eps)
    test_normal_blocks  = (test_normal_blocks - mu) / (sig + eps)
    if test_anomaly_blocks.shape[0] > 0:
        test_anomaly_blocks = (test_anomaly_blocks - mu) / (sig + eps)
    
    print(f"  Post-norm train range: [{train_blocks.min():.2f}, {train_blocks.max():.2f}]")
    
    # --- Save to disk ---
    np.save(str(out_dir / "train_normal.npy"), train_blocks)
    np.save(str(out_dir / "val_normal.npy"), val_blocks)
    np.save(str(out_dir / "test_normal.npy"), test_normal_blocks)
    np.save(str(out_dir / "test_anomaly.npy"), test_anomaly_blocks)
    np.savez(str(out_dir / "norm_stats.npz"), mean=train_mean, std=train_std)
    
    print(f"  ✓ Saved to: {out_dir}")
    
    # --- Record summary ---
    pipeline_summary.append({
        "machine": machine,
        "train_clips": len(train_files),
        "val_clips": len(val_files),
        "anomaly_clips": len(anomaly_files),
        "train_blocks": train_blocks.shape[0],
        "val_blocks": val_blocks.shape[0],
        "test_normal_blocks": test_normal_blocks.shape[0],
        "test_anomaly_blocks": test_anomaly_blocks.shape[0],
        "extraction_sec": round(elapsed, 1),
    })
    
    # --- Free memory ---
    del train_blocks, val_blocks, test_normal_blocks, test_anomaly_blocks
    gc.collect()

print("\n\n" + "="*70)
print("  PREPROCESSING COMPLETE")
print("="*70)

---
### Step 8: Preprocessing Summary Report

This cell compiles and displays a summary table of the entire preprocessing run:
- Number of source clips and extracted blocks per machine type and partition.
- Extraction time.
- Total blocks available for training.

The summary is also saved as `reports/preprocessing_summary.csv` for reference.

In [ ]:
if pipeline_summary:
    df_summary = pd.DataFrame(pipeline_summary)
    df_summary["total_blocks"] = (
        df_summary["train_blocks"] + df_summary["val_blocks"] + 
        df_summary["test_normal_blocks"] + df_summary["test_anomaly_blocks"]
    )
    
    print("\nPreprocessing Summary:")
    display(df_summary)
    
    # Save summary
    summary_path = os.path.join(OUTPUT_DIR, "preprocessing_summary.csv")
    df_summary.to_csv(summary_path, index=False)
    print(f"\n✓ Summary saved: {summary_path}")
    
    # Grand totals
    total_train = df_summary["train_blocks"].sum()
    total_val   = df_summary["val_blocks"].sum()
    total_test  = df_summary["test_normal_blocks"].sum() + df_summary["test_anomaly_blocks"].sum()
    total_time  = df_summary["extraction_sec"].sum()
    
    print(f"\n  Grand Total Training Blocks:   {total_train:,}")
    print(f"  Grand Total Validation Blocks: {total_val:,}")
    print(f"  Grand Total Test Blocks:       {total_test:,}")
    print(f"  Total Extraction Time:         {total_time:.0f}s ({total_time/60:.1f} min)")
else:
    print("⚠ No machines were processed.")

---
### Step 9: Post-Processing Verification — Reload & Validate Saved Tensors

As a final quality gate, this cell **reloads** the saved `.npy` files from disk and checks:
1. **Shape correctness:** Every tensor must have 4 dimensions `(N, 1, 128, 5)`.
2. **No NaN or Inf values:** These would crash the Autoencoder during training.
3. **Normalisation quality:** After Z-score normalisation, the training data should have approximately zero mean and unit standard deviation per frequency bin.
4. **File sizes:** Printed for disk-space awareness (important on Kaggle's ~20 GB output limit).

This is a non-negotiable verification step — catching a corrupt tensor *now* saves hours of debugging a training crash later.

In [ ]:
print("Post-Processing Verification")
print("=" * 60)

all_ok = True

for machine in PREPROCESS_CFG["machines"]:
    out_dir = PROCESSED_DIR / machine
    print(f"\n▶ {machine.upper()}:")
    
    for split_name in ["train_normal", "val_normal", "test_normal", "test_anomaly"]:
        npy_path = out_dir / f"{split_name}.npy"
        if not npy_path.exists():
            print(f"    ✗ MISSING: {npy_path}")
            all_ok = False
            continue
        
        data = np.load(str(npy_path))
        size_mb = npy_path.stat().st_size / (1024 * 1024)
        has_nan = np.isnan(data).any()
        has_inf = np.isinf(data).any()
        
        status = "✓" if (not has_nan and not has_inf and len(data.shape) == 4) else "✗"
        if status == "✗":
            all_ok = False
        
        print(f"    {status} {split_name:18s}: shape={str(data.shape):25s} range=[{data.min():.2f}, {data.max():.2f}]  NaN={has_nan}  Inf={has_inf}  size={size_mb:.1f} MB")
    
    # Check norm stats
    stats_path = out_dir / "norm_stats.npz"
    if stats_path.exists():
        stats = np.load(str(stats_path))
        print(f"    ✓ norm_stats.npz: mean shape={stats['mean'].shape}, std shape={stats['std'].shape}")
    else:
        print(f"    ✗ MISSING: norm_stats.npz")
        all_ok = False

print(f"\n{'='*60}")
if all_ok:
    print("✓ ALL VERIFICATIONS PASSED — tensors are ready for Autoencoder training!")
else:
    print("✗ SOME VERIFICATIONS FAILED — check the errors above.")

---
### Step 10: Visual Verification — Before & After Normalisation

As a final visual confirmation, we plot a sample context block **before** and **after** Z-score normalisation for one machine type. This lets you visually confirm that:
- The normalised block has roughly zero-centred pixel values.
- The frequency-bin structure (horizontal patterns) is preserved.
- No frequency band is disproportionately amplified or crushed.

---

### Conclusion & Next Steps
- **Phase 2 is now complete:** Raw audio has been transformed into normalised, GPU-ready tensor blocks.
- **Proceed to Phase 3 (Notebook 04):** Build the Conv2D Autoencoder architecture and train it on the `train_normal.npy` blocks.

In [ ]:
# Reload one machine's data for visual comparison
viz_machine = PREPROCESS_CFG["machines"][0]
viz_dir = PROCESSED_DIR / viz_machine

if (viz_dir / "train_normal.npy").exists() and (viz_dir / "norm_stats.npz").exists():
    train_data = np.load(str(viz_dir / "train_normal.npy"))
    stats = np.load(str(viz_dir / "norm_stats.npz"))
    
    # Pick a random block
    np.random.seed(42)
    idx = np.random.randint(0, train_data.shape[0])
    normalised_block = train_data[idx, 0]  # shape: (128, 5)
    
    # Reverse the normalisation to get the "before" view
    mu = stats['mean'].reshape(-1, 1)
    sig = stats['std'].reshape(-1, 1)
    original_block = normalised_block * (sig + PREPROCESS_CFG['epsilon']) + mu
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    im0 = axes[0].imshow(original_block, aspect='auto', origin='lower', cmap='magma')
    axes[0].set_title(f'Before Normalisation (dB scale)\n{viz_machine}', fontweight='bold')
    axes[0].set_xlabel('Context Frame')
    axes[0].set_ylabel('Mel Bin')
    fig.colorbar(im0, ax=axes[0], format='%.0f')
    
    im1 = axes[1].imshow(normalised_block, aspect='auto', origin='lower', cmap='coolwarm', vmin=-3, vmax=3)
    axes[1].set_title(f'After Z-Score Normalisation\n{viz_machine}', fontweight='bold')
    axes[1].set_xlabel('Context Frame')
    axes[1].set_ylabel('Mel Bin')
    fig.colorbar(im1, ax=axes[1], format='%.1f')
    
    plt.suptitle('Preprocessing Verification: Before vs. After Normalisation', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    viz_path = os.path.join(OUTPUT_DIR, "preprocessing_verification.png")
    plt.savefig(viz_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✓ Verification plot saved: {viz_path}")
    
    del train_data
    gc.collect()
else:
    print("⚠ Processed data not found for visual verification.")

print("\n" + "="*60)
print("PHASE 2 — NOTEBOOK 03 COMPLETED SUCCESSFULLY!")
print("="*60)